# Algions AI Labs - Pharmacy Medicine Demand & Expiry-Risk Forecasting
### End-to-End Exploratory Data Analysis, Leakage Verification, and Model Benchmarks

**Mission**: Build a decision-support ML solution with reproducible data preparation, defensible evaluation, and a working inference interface.

> **Operational Advisory Notice**: All predictions and alerts are intended solely as supply-chain and inventory triage guidance for pharmacy inventory managers. They never constitute clinical, dosage, or medical treatment advice.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set seed for absolute reproducibility
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Data Ingestion & Schema Validation

In [ ]:
from src.data_loader import PharmaDataLoader

loader = PharmaDataLoader("data/processed/pharmacy_demand_expiry_data.csv")
raw_df = loader.load_and_validate()
stats = loader.get_summary_statistics()

print(f"Dataset Summary:")
print(f"  Total Records: {stats['total_records']}")
print(f"  Products: {stats['num_products']} | Batches: {stats['num_batches']}")
print(f"  Date Span: {stats['date_range'][0]} to {stats['date_range'][1]}")
print(f"  Total Units Sold: {stats['total_units_sold']:,}")
raw_df.head()

## 2. Exploratory Data Analysis (EDA)
- Demand distributions across drug categories
- Seasonality trends (monsoon spikes for analgesics, winter peaks for antibiotics)
- Days to Expiry vs Remaining Stock Balance

In [ ]:
# Aggregate weekly demand per product
weekly_df = loader.aggregate_weekly_demand()

plt.figure(figsize=(14, 5))
for prod_id, grp in weekly_df.groupby("Product_ID"):
    plt.plot(grp["Date"], grp["Total_Units_Sold"], label=f"{prod_id} ({grp['Product_Name'].iloc[0]})")
plt.title("Weekly Medicine Demand Trajectory Over Time", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Strict Anti-Leakage Feature Engineering
All lag and rolling features are shifted by **at least 1 period** (`shift(1)`), ensuring that the model at week $T$ only sees information up to week $T-1$.

In [ ]:
from src.features import FeatureEngineer

fe = FeatureEngineer()
feat_df = fe.create_demand_features(weekly_df, target_col="Total_Units_Sold")
feature_cols = fe.get_feature_names(feat_df)

print(f"Engineered {len(feature_cols)} Leakage-Free Features:")
print(feature_cols)

## 4. Chronological Split (Train 70% / Validation 15% / Held-Out Test 15%)

In [ ]:
train_df, val_df, test_df = loader.chronological_split(feat_df, train_ratio=0.70, val_ratio=0.15)

print(f"Train dates: {train_df['Date'].min().date()} to {train_df['Date'].max().date()} ({len(train_df)} rows)")
print(f"Val dates  : {val_df['Date'].min().date()} to {val_df['Date'].max().date()} ({len(val_df)} rows)")
print(f"Test dates : {test_df['Date'].min().date()} to {test_df['Date'].max().date()} ({len(test_df)} rows)")

# Verification: Zero overlap
assert train_df['Date'].max() < val_df['Date'].min()
assert val_df['Date'].max() < test_df['Date'].min()
print("VERIFIED: Zero chronological lookahead leakage.")

## 5. Model Training & Evaluation on Held-Out Test Set

In [ ]:
from src.train import run_pipeline

metrics_report = run_pipeline(
    data_path="data/processed/pharmacy_demand_expiry_data.csv",
    output_dir="models",
    seed=42
)

## 6. Batch Expiry Triage Inspection
Simulate stock balance triage using the trained risk engine.

In [ ]:
from src.models.risk_classifier import InventoryRiskEngine

engine = InventoryRiskEngine(risk_horizon_days=60)
sample_batch = {
    "Batch_ID": "BAT_001_CRIT",
    "Product_ID": "PROD_001",
    "Product_Name": "Amoxicillin 500mg",
    "Stock_Balance": 420,
    "Days_To_Expiry": 30,
    "Unit_Price": 12.50
}

triage_result = engine.evaluate_batch(sample_batch, forecasted_weekly_demand=15.0)
for k, v in triage_result.items():
    print(f"{k}: {v}")